In [1]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain_chroma import Chroma

import numpy as np
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains.retrieval import create_retrieval_chain

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [2]:
loader = DirectoryLoader(
    path="../data/txt",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_documents(documents=documents)

store_location = "../vector_store/chroma_db"
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')


vector_store = Chroma.from_documents(
    documents=chunks,
    collection_name="memory_eval_store1",
    embedding=embeddings_model,
    persist_directory=store_location
)

print(f"Number of vectors created: {vector_store._collection.count()}")
retriever = vector_store.as_retriever()

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 96.28it/s]


Number of vectors created: 46


In [3]:
llm = ChatOpenAI(model_name='gpt-4o-mini', max_tokens=500)

# 1. Using create_history_aware_retriever from Langchain (Legacy)

In [5]:
# Create a prompt that includes chat history

system_prompt_reformulation_instruction = """Given a chat history and the latest user question which might reference 
context in the chat history, formulate a standalone question which can be understood
without the chat history. DO NOT answer the question, just reforumlate it if needed
otherwise return it as it is."""

reformulation_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_reformulation_instruction),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [6]:
# create history aware retriever

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    reformulation_prompt
)

In [7]:
system_prompt = """You are a helpful assistant for question answering tasks. 
Use the following pieces of retrieved context to answer the question.
If you dont know the answer simply say I dont know. Use maximum 3 sentences for the response

Context: {context}"""


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

chain = create_stuff_documents_chain(llm, prompt)


In [8]:
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    chain
)

In [9]:
chat_history = []

result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})

In [13]:
input, output = result1['input'], result1['answer']

print(input)
print(output)

What is machine learning?
Machine learning (ML) is a type of artificial intelligence that enables software to learn from data in order to make predictions or decisions without being explicitly programmed. It involves algorithms that improve their performance as they are exposed to more data. The main types of machine learning include supervised, unsupervised, semi-supervised, and reinforcement learning.


In [15]:
chat_history.extend([
    HumanMessage(content=input),
    AIMessage(content=output)
])

chat_history

[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning (ML) is a type of artificial intelligence that enables software to learn from data in order to make predictions or decisions without being explicitly programmed. It involves algorithms that improve their performance as they are exposed to more data. The main types of machine learning include supervised, unsupervised, semi-supervised, and reinforcement learning.', additional_kwargs={}, response_metadata={})]

In [16]:
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its types?"
})

input, output = result2['input'], result2['answer']

print(input)
print(output)

What are its types?
The main types of machine learning are supervised learning, unsupervised learning, semi-supervised learning, and reinforcement learning. Each type is suited for different problems and data types. Supervised learning involves learning from labeled data, unsupervised learning deals with unlabeled data, semi-supervised learning is a mix of both, and reinforcement learning focuses on learning through interaction with an environment.


In [17]:
chat_history.extend([
    HumanMessage(content=input),
    AIMessage(content=output)
])

chat_history

[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning (ML) is a type of artificial intelligence that enables software to learn from data in order to make predictions or decisions without being explicitly programmed. It involves algorithms that improve their performance as they are exposed to more data. The main types of machine learning include supervised, unsupervised, semi-supervised, and reinforcement learning.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What are its types?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The main types of machine learning are supervised learning, unsupervised learning, semi-supervised learning, and reinforcement learning. Each type is suited for different problems and data types. Supervised learning involves learning from labeled data, unsupervised learning deals with unlabeled data, semi-supervised learning is a mix of both, a